# Chapter 5 — Singleton Pattern
## *One-of-a-Kind Objects*

**Intent:** Ensure a class has only **one instance**, and provide a global point of access to it.

### Book context
Examples: chocolate boiler, thread pool, registry, logger, configuration.  
The book shows classic Java double-checked locking and its pitfalls — in Python we have cleaner options.

### Warning
Singleton is the most **controversial** design pattern.  
It introduces global state and makes testing harder. Prefer dependency injection when possible.

## Implementation 1 — `__new__` (classic)

In [ ]:
class Singleton:
    _instance = None

    def __new__(cls, *args, **kwargs):
        if cls._instance is None:
            cls._instance = super().__new__(cls)
        return cls._instance


class ChocolateBoiler(Singleton):
    def __init__(self):
        # Guard: __init__ runs every call to ChocolateBoiler()
        if not hasattr(self, "_initialized"):
            self._empty = True
            self._boiled = False
            self._initialized = True

    def fill(self):
        if self._empty:
            self._empty = False
            self._boiled = False
            print("Filling boiler with milk/chocolate mix.")

    def drain(self):
        if not self._empty and self._boiled:
            self._empty = True
            print("Draining boiler.")

    def boil(self):
        if not self._empty and not self._boiled:
            self._boiled = True
            print("Boiling contents.")


b1 = ChocolateBoiler()
b2 = ChocolateBoiler()
print(f"Same instance? {b1 is b2}")   # True
b1.fill()
b2.boil()  # works on the same object

## Implementation 2 — Metaclass (Pythonic)

In [ ]:
class SingletonMeta(type):
    _instances: dict = {}

    def __call__(cls, *args, **kwargs):
        if cls not in cls._instances:
            cls._instances[cls] = super().__call__(*args, **kwargs)
        return cls._instances[cls]


class Logger(metaclass=SingletonMeta):
    def __init__(self):
        self._logs: list[str] = []

    def log(self, msg: str):
        self._logs.append(msg)
        print(f"[LOG] {msg}")

    def history(self):
        return list(self._logs)


l1 = Logger()
l2 = Logger()
l1.log("App started")
l2.log("User logged in")
print(f"Same? {l1 is l2}")
print("History:", l1.history())  # both logs visible

## Implementation 3 — Module-level (most Pythonic)

Python modules are singletons by design — imported once and cached in `sys.modules`.

In [ ]:
# config_singleton.py  (simulated here as a module-level object)
class _Config:
    def __init__(self):
        self.db_url = "postgresql://localhost/mydb"
        self.debug  = False

config = _Config()  # created once at import time

# Usage from other modules: `from config_singleton import config`
print(config.db_url)

## Thread-safe Singleton

In [ ]:
import threading

class ThreadSafeSingleton:
    _instance = None
    _lock = threading.Lock()

    def __new__(cls):
        if cls._instance is None:          # first check (no lock)
            with cls._lock:                # acquire lock
                if cls._instance is None:  # second check (with lock)
                    cls._instance = super().__new__(cls)
        return cls._instance


def create_instance():
    s = ThreadSafeSingleton()
    print(f"Thread {threading.current_thread().name}: id={id(s)}")

threads = [threading.Thread(target=create_instance) for _ in range(4)]
for t in threads: t.start()
for t in threads: t.join()

## Interview cheat-sheet

| Question | Answer |
|---|---|
| When to use? | Logger, config, connection pool, device driver, cache. |
| Drawbacks? | Global state, hard to unit-test, violates SRP (manages its own lifecycle). |
| Thread safety approach? | Double-checked locking; in Python, metaclass or module-level is simpler and naturally thread-safe at import. |
| Alternative? | Dependency Injection — pass the shared object as a parameter instead of using a global. |

**Pattern family:** Creational